<div align="center">

# Day 04: Reliability of Explanations

**Explainable Artificial Intelligence (VTR UGE 21), hands-on notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Day overview](https://github.com/utkukose/xai-veltech-NB-lecture/tree/main/days/day-04) · [Lecture page](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-04/lecture.html) · [Interactive lab](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-04/lab.html) · [PDF notes](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-04/Day04_Lecture_Notes.pdf)

</div>

## How to use this notebook

This notebook is the hands-on part of the day. The concepts are explained on the [lecture page](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-04/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Nothing needs to be installed.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback.

In [ ]:
# Environment: Colab installs the three explanation libraries; everything else is preinstalled.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'shap', 'lime', 'gemex'], check=False)

import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (7, 3.6), 'axes.grid': True, 'grid.alpha': 0.3})

def check(name, value, expected, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback."""
    if value is None:
        print(f"{name}: not answered yet. Replace None with your answer and run the cell again.")
        return False
    try:
        ok = abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected)))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else "not yet. Compare your steps with the lecture and try again."))
    return ok


print('ready')

## Python step 4: Data frames and plots

Audits of models are mostly operations on tables: Filtering, grouping and binning. This step does each of them with pandas on a handful of credit applicants and draws a reliability table &#91;[6](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-04/lecture.html#ref-6), [7](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-04/lecture.html#ref-7)&#93;.

### A data frame and its summary

In [ ]:
import pandas as pd
credit = pd.DataFrame({"income": [42000, 31000, 58000, 27000, 39000, 51000],
                       "late_payments": [0, 3, 1, 4, 2, 0],
                       "approved": [1, 0, 1, 0, 1, 1]})
print(credit)
print(credit.describe().round(1))

A data frame holds named columns of equal length. `describe` summarises every numerical column: The count, the mean, the standard deviation and the quartiles.

### Filtering rows

In [ ]:
late = credit[credit["late_payments"] >= 2]
print(late)
print(credit.loc[credit["income"] > 40000, ["late_payments", "approved"]])

A comparison with a column gives a column of booleans, and indexing the data frame with it keeps the rows where it is `True`. `loc` selects rows and columns at once, by condition and by name.

### Grouping

In [ ]:
credit["many_late"] = credit["late_payments"] >= 2
rates = credit.groupby("many_late")["approved"].mean()
print(rates)
print("difference of the approval shares:", round(float(rates.max() - rates.min()), 3))

A new column is created by assigning to a new name, as in a dictionary. `groupby` splits the rows by the values of a column, here whether an applicant has two or more late payments, and the mean of the approval column in each group is the share of approved applicants. All applicants with fewer late payments were approved, and one of the three with more.

### A reliability table

In [ ]:
probs = pd.Series([0.05, 0.12, 0.35, 0.41, 0.62, 0.68, 0.81, 0.93, 0.97, 0.22])
outcome = pd.Series([0, 0, 1, 0, 1, 1, 1, 1, 1, 0])
bins = pd.cut(probs, [0, 0.25, 0.5, 0.75, 1.0])
table = pd.DataFrame({"confidence": probs, "outcome": outcome}).groupby(bins, observed=True).mean()
print(table)

`pd.cut` assigns every probability to an interval, and grouping by the intervals gives the mean confidence and the observed frequency of each bin, the two coordinates of a reliability diagram.

### A plot from a data frame

In [ ]:
import matplotlib.pyplot as plt
ax = table.plot(x="confidence", y="outcome", marker="o", legend=False, figsize=(4.5, 3.2))
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_xlabel("mean confidence"); ax.set_ylabel("observed frequency")
plt.show()

Data frames can plot themselves: `plot` returns the axes, on which further lines and labels can be drawn. The dashed diagonal marks perfect calibration.

**Quick check.** Why can the area under the ROC curve not detect miscalibration?

<details><summary>Answer</summary>

Because it depends only on the ranking of the scores: Changing the scores in a monotone way, which changes calibration, leaves the ranking and the area unchanged.

</details>

### Exercises for Python step 4

**Exercise 4.1.** Store the difference between the two approval shares, rounded to three decimals, in `gap`.

In [ ]:
gap = None   # your answer

check("Exercise 4.1", gap, 0.667)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
gap = round(float(rates.max() - rates.min()), 3)

check("Exercise 4.1", gap, 0.667)

**Exercise 4.2.** How many applicants have two or more late payments? Store the number in `n_late`.

In [ ]:
n_late = None   # your answer

check("Exercise 4.2", n_late, 3)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_late = len(late)

check("Exercise 4.2", n_late, 3)

**Exercise 4.3.** Store the observed frequency of the highest bin, the last value of `table["outcome"]`, in `top_freq`.

In [ ]:
top_freq = None   # your answer

check("Exercise 4.3", top_freq, 1.0)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
top_freq = float(table["outcome"].iloc[-1])

check("Exercise 4.3", top_freq, 1.0)

## 1. Can the probability be trusted? Calibration &#91;[1](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-04/lecture.html#ref-1)&#93;

*Lecture: [Can the probability be trusted?](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-04/lecture.html#can-the-probability-be-trusted)*

Two datasets: the tumour data of the week and synthetic credit data with a known proxy.

**From the formulas to the code.** `make_credit` draws the six features and the label `repaid`. The number that the first cell prints as approval rate, 0.31, is the mean of this label: the share of applicants who repay. In `ece`, the function for the expected calibration error (ECE), `idx` is the bin of every case, `m.mean()` the share $n_b / n$, `p[m].mean()` the mean predicted probability $\bar{p}_b$ and `y_true[m].mean()` the observed frequency $\bar{y}_b$. Python counts the ten bins from 0, a probability of exactly 1 goes into the last bin, and empty bins are skipped. In the table, `AUC` is the area under the receiver operating characteristic (ROC) curve from `roc_auc_score`, `Brier` is the Brier score, and `ECE_5` and `ECE_95` are the 5th and 95th percentile of the ECE over the 200 bootstrap resamples whose row numbers are stored in `boot`. `CalibratedClassifierCV` with `method="isotonic"` and `cv=5` fits the isotonic map on five splits of the training data and averages the five corrected models.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
data = load_breast_cancer(as_frame=True)
FEATURES = list(data.feature_names)
Xtr, Xte, ytr, yte = train_test_split(data.data, data.target, test_size=0.25, stratify=data.target, random_state=RANDOM_STATE)
# ---------------------------------------------------------------- Dataset B (synthetic credit, known bias)
def make_credit(n=4000, seed=RANDOM_STATE, bias_strength=1.2):
    rng = np.random.default_rng(seed)
    group = rng.choice(["A", "B"], size=n, p=[0.65, 0.35]); is_b = (group == "B").astype(float)
    income        = np.exp(rng.normal(10.4 - 0.20 * is_b, 0.45, n))
    debt_ratio    = np.clip(rng.beta(2, 5, n) + 0.05 * is_b, 0, 1)
    history_years = np.clip(rng.gamma(3.0, 2.2, n), 0, 40)
    late_payments = rng.poisson(1.3 + 0.5 * is_b, n)
    employment    = np.clip(rng.gamma(2.5, 2.0, n), 0, 35)
    postcode      = np.clip(rng.normal(60 - 18 * is_b, 12, n), 0, 100)   # proxy for group
    logit = (0.9 * (np.log(income) - 10.4) - 2.4 * debt_ratio + 0.045 * history_years
             - 0.30 * late_payments + 0.035 * employment
             + bias_strength * (postcode - 60) / 40.0)
    y = rng.binomial(1, 1 / (1 + np.exp(-logit)))
    X = pd.DataFrame({"income": income.round(0), "debt_ratio": debt_ratio.round(3),
                      "history_years": history_years.round(1), "late_payments": late_payments,
                      "employment_years": employment.round(1), "postcode_score": postcode.round(1)})
    return X, pd.Series(y, name="repaid"), pd.Series(group, name="group")

X_cr, y_cr, g_cr = make_credit()
Xtr_c, Xte_c, ytr_c, yte_c, gtr, gte = train_test_split(X_cr, y_cr, g_cr, test_size=0.3, stratify=y_cr, random_state=RANDOM_STATE)
print(f"tumour data {data.data.shape}; credit data {X_cr.shape}, approval rate {y_cr.mean():.2f}, share of group B {(g_cr == 'B').mean():.2f}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, brier_score_loss

def ece(y_true, p, bins=10):
    """Expected calibration error: the share-weighted gap between confidence and frequency in each bin."""
    idx = np.clip((p * bins).astype(int), 0, bins - 1); e, xs, ys = 0.0, [], []
    for b in range(bins):
        m = idx == b
        if m.any():
            e += m.mean() * abs(y_true[m].mean() - p[m].mean()); xs.append(p[m].mean()); ys.append(y_true[m].mean())
    return e, np.array(xs), np.array(ys)

models = {"gradient boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
          "deep gradient boosting": GradientBoostingClassifier(max_depth=6, n_estimators=400, learning_rate=0.2,
                                                               random_state=RANDOM_STATE),
          "naive Bayes": GaussianNB()}
boot = np.random.default_rng(RANDOM_STATE).integers(0, len(yte_c), (200, len(yte_c)))   # bootstrap resamples
rows, curves = [], {}
for name, m in models.items():
    for kind, est in (("raw", m), ("isotonic", CalibratedClassifierCV(m, method="isotonic", cv=5))):
        p = est.fit(Xtr_c, ytr_c).predict_proba(Xte_c)[:, 1]
        e, xs, ys = ece(yte_c.values, p); curves[(name, kind)] = (xs, ys)
        eb = [ece(yte_c.values[b], p[b])[0] for b in boot]
        rows.append(dict(model=name, version=kind, AUC=roc_auc_score(yte_c, p), Brier=brier_score_loss(yte_c, p), ECE=e,
                         ECE_5=np.percentile(eb, 5), ECE_95=np.percentile(eb, 95)))
calib = pd.DataFrame(rows)
print(calib.round(4).to_string(index=False))

fig, ax = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
for a, name in zip(ax, models):
    a.plot([0, 1], [0, 1], "k--", lw=1)
    for kind, col in (("raw", "#c0392b"), ("isotonic", "#2980b9")):
        xs, ys = curves[(name, kind)]; a.plot(xs, ys, "o-", ms=3, color=col, label=kind)
    a.set_title(name, fontsize=9); a.set_xlabel("mean predicted probability")
ax[0].set_ylabel("observed frequency"); ax[0].legend(fontsize=8); plt.tight_layout(); plt.show()
gb = models["gradient boosting"]                                      # the model audited in the rest of the notebook

**Exercise A.** Which model has the largest calibration error before recalibration? Store its name in `worst_model`.

In [ ]:
worst_model = None   # your answer

check("Exercise A", worst_model, calib[calib.version == "raw"].set_index("model").ECE.idxmax())

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
worst_model = calib[calib.version == "raw"].set_index("model").ECE.idxmax()

check("Exercise A", worst_model, calib[calib.version == "raw"].set_index("model").ECE.idxmax())

## 2. Has the population changed?

*Lecture: [Has the population changed?](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-04/lecture.html#has-the-population-changed)*

**From the formulas to the code.** `yd` is the domain label: 0 for the 2800 training applicants and 1 for the 1500 new ones. `cross_val_predict` with `cv=5` returns for every case the probability from a forest that was trained without it, and `roc_auc_score(yd, p_dom)` is the area under the receiver operating characteristic curve (AUC) of the lecture. `dom.feature_importances_` holds the impurity importances, which add up to 1. The three arguments of `assign` are the simulated downturn: incomes times 0.82, one more late payment, and debt ratios times 1.15, which `np.clip` keeps at 1 or below. The line printed as which features moved is the ranking by impurity importance, which the lecture reads with care.

In [ ]:
# Simulate a shifted population: economic downturn, incomes fall, late payments rise
X_new, y_new, g_new = make_credit(n=1500, seed=99)
X_new = X_new.assign(income=X_new.income * 0.82,
                     late_payments=X_new.late_payments + 1,
                     debt_ratio=np.clip(X_new.debt_ratio * 1.15, 0, 1))

Xd = pd.concat([Xtr_c, X_new], ignore_index=True)
yd = np.r_[np.zeros(len(Xtr_c)), np.ones(len(X_new))]
dom = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1)
from sklearn.model_selection import cross_val_predict
p_dom = cross_val_predict(dom, Xd, yd, cv=5, method="predict_proba")[:, 1]
auc_dom = roc_auc_score(yd, p_dom)

dom.fit(Xd, yd)
share = pd.Series(dom.feature_importances_, index=X_cr.columns).sort_values(ascending=False)

print(f"domain-classifier AUC = {auc_dom:.3f}")
print("which features moved:")
print(share.round(3).to_string())

print(f"\nmodel performance:")
print(f"   original test set : auc {roc_auc_score(yte_c, models['gradient boosting'].predict_proba(Xte_c)[:, 1]):.3f}")
print(f"   shifted population: auc {roc_auc_score(y_new, models['gradient boosting'].predict_proba(X_new)[:, 1]):.3f}")

In [ ]:
moved = pd.Series(dom.feature_importances_, index=X_cr.columns).sort_values()
moved.plot.barh(figsize=(6, 3), color="#2980b9"); plt.xlabel("importance in the domain classifier")
plt.title("which features changed between the two populations", fontsize=9); plt.show()

## 3. A shortcut network on three slices

*Lecture: [Has the population changed?](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-04/lecture.html#has-the-population-changed)*

**From the formulas to the code.** In `make_lesion_images`, `marker_rate` is the probability that an image with a lesion carries the marker, and an image without a lesion carries it with the probability `1 - marker_rate`. `small_cnn` builds the convolutional neural network (CNN) of Day 3. `cnn.forward` returns the two scores $s_0(x)$ and $s_1(x)$ of every image, and `softmax_rows` turns them into the probabilities `P`. `P.argmax(1)` is the predicted class and `P.max(1)` its confidence, so the two columns of the table are the accuracy and the mean confidence of the lecture. All three slices use `seed=5` and therefore contain the same 600 images, apart from the marker.

In [ ]:
#@title A small neural network written with NumPy (run this cell; open it to read the code)
# A small neural-network kit written with NumPy. Every layer has a forward pass that stores what the backward pass
# needs, and a backward pass that returns the gradient with respect to its input and keeps the gradients of its
# weights. Deep-learning libraries automate exactly these steps.

def im2col(X, k=3, pad=1):
    """Patches of a stride-1 convolution with 'same' padding: (N, C, H, W) -> (N*H*W, C*k*k)."""
    N, C, H, W = X.shape
    Xp = np.pad(X, ((0, 0), (0, 0), (pad, pad), (pad, pad)))
    cols = np.empty((N, C, k, k, H, W), dtype=X.dtype)
    for i in range(k):
        for j in range(k):
            cols[:, :, i, j] = Xp[:, :, i:i + H, j:j + W]
    return cols.transpose(0, 4, 5, 1, 2, 3).reshape(N * H * W, C * k * k)

def col2im(cols, shape, k=3, pad=1):
    """The adjoint of im2col: adds every patch gradient back to its pixels."""
    N, C, H, W = shape
    cols = cols.reshape(N, H, W, C, k, k).transpose(0, 3, 4, 5, 1, 2)
    Xp = np.zeros((N, C, H + 2 * pad, W + 2 * pad), dtype=cols.dtype)
    for i in range(k):
        for j in range(k):
            Xp[:, :, i:i + H, j:j + W] += cols[:, :, i, j]
    return Xp[:, :, pad:pad + H, pad:pad + W]

class Conv:
    def __init__(self, cin, cout, rng):
        self.W = rng.normal(0, np.sqrt(2 / (cin * 9)), (cin * 9, cout)); self.b = np.zeros(cout)
    def forward(self, X):
        N, C, H, W = X.shape; self.shape = X.shape; self.cols = im2col(X)
        return (self.cols @ self.W + self.b).reshape(N, H, W, -1).transpose(0, 3, 1, 2)
    def backward(self, G):
        Gm = G.transpose(0, 2, 3, 1).reshape(-1, G.shape[1])
        self.dW, self.db = self.cols.T @ Gm, Gm.sum(0)
        return col2im(Gm @ self.W.T, self.shape)

class ReLU:
    def forward(self, X):
        self.m = X > 0; return X * self.m
    def backward(self, G):
        return G * self.m

class MaxPool:
    def forward(self, X):
        N, C, H, W = X.shape; Xr = X.reshape(N, C, H // 2, 2, W // 2, 2); out = Xr.max(axis=(3, 5))
        m = Xr == out[:, :, :, None, :, None]
        self.mask = m / m.sum(axis=(3, 5), keepdims=True); self.shape = X.shape
        return out
    def backward(self, G):
        return (self.mask * G[:, :, :, None, :, None]).reshape(self.shape)

class Flatten:
    def forward(self, X):
        self.shape = X.shape; return X.reshape(len(X), -1)
    def backward(self, G):
        return G.reshape(self.shape)

class Dense:
    def __init__(self, din, dout, rng, bias=True):
        self.W = rng.normal(0, np.sqrt(2 / din), (din, dout)); self.b = np.zeros(dout); self.bias = bias
    def forward(self, X):
        self.X = X; return X @ self.W + self.b
    def backward(self, G):
        self.dW = self.X.T @ G; self.db = G.sum(0) if self.bias else np.zeros_like(self.b)
        return G @ self.W.T

class Net:
    """A sequence of layers with a forward pass, a backward pass and Adam."""
    def __init__(self, layers):
        self.layers = layers
        self.state = [(np.zeros_like(p), np.zeros_like(p)) for p in self.params()]; self.t = 0
    def params(self):
        return [p for L in self.layers if hasattr(L, "W") for p in (L.W, L.b)]
    def grads(self):
        return [g for L in self.layers if hasattr(L, "W") for g in (L.dW, L.db)]
    def forward(self, X):
        self.acts = []
        for L in self.layers:
            X = L.forward(X); self.acts.append(X)
        return X
    def backward(self, G):
        """G is the gradient with respect to the output. Stores the gradient at every layer output in self.dacts."""
        self.dacts = [None] * len(self.layers)
        for i in reversed(range(len(self.layers))):
            self.dacts[i] = G; G = self.layers[i].backward(G)
        return G
    def adam_step(self, lr=2e-3):
        self.t += 1
        for k, (p, g) in enumerate(zip(self.params(), self.grads())):
            m, v = self.state[k]; m[...] = 0.9 * m + 0.1 * g; v[...] = 0.999 * v + 0.001 * g * g
            p -= lr * (m / (1 - 0.9 ** self.t)) / (np.sqrt(v / (1 - 0.999 ** self.t)) + 1e-8)

def softmax_rows(Z):
    E = np.exp(Z - Z.max(1, keepdims=True)); return E / E.sum(1, keepdims=True)

def train_classifier(net, X, y, epochs=8, batch=128, lr=2e-3, seed=0):
    """Mini-batch training with the cross-entropy loss; returns the loss of every epoch."""
    rng = np.random.default_rng(seed); losses = []
    for ep in range(epochs):
        perm = rng.permutation(len(X)); tot = 0.0
        for b in range(0, len(X), batch):
            idx = perm[b:b + batch]; P = softmax_rows(net.forward(X[idx]))
            tot += -np.log(P[np.arange(len(idx)), y[idx]] + 1e-12).sum()
            G = P.copy(); G[np.arange(len(idx)), y[idx]] -= 1.0
            net.backward(G / len(idx)); net.adam_step(lr)
        losses.append(tot / len(X))
    return losses

def small_cnn(seed=0):
    rng = np.random.default_rng(seed)
    return Net([Conv(1, 8, rng), ReLU(), MaxPool(), Conv(8, 16, rng), ReLU(), MaxPool(),
                Conv(16, 32, rng), ReLU(), Flatten(), Dense(32 * 8 * 8, 2, rng)])

CAM_LAYER = 7     # the ReLU after the last convolution: 32 maps of 8 x 8, used by Grad-CAM and TCAV

def output_gradient(out, target=None):
    """Seed of the backward pass: the log-odds of class 1 if target is None, otherwise one logit."""
    G = np.zeros_like(out)
    if target is None:
        G[:, 1], G[:, 0] = 1.0, -1.0
    else:
        G[:, target] = 1.0
    return G

def input_gradient(net, X, target=None):
    """Gradient with respect to the input of the log-odds of class 1, or of the logit `target`."""
    return net.backward(output_gradient(net.forward(X), target))

In [ ]:
def make_lesion_images(n=3000, marker_rate=0.97, seed=RANDOM_STATE, size=32):
    rng = np.random.default_rng(seed)
    X = rng.normal(0.30, 0.08, size=(n, size, size)).astype("float32"); y = rng.integers(0, 2, n)
    yy, xx = np.mgrid[0:size, 0:size]
    for i in range(n):
        for _ in range(6):
            cy, cx = rng.integers(4, size - 4, 2); s = rng.uniform(2.5, 6)
            X[i] += rng.uniform(0.10, 0.22) * np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * s ** 2))
        if y[i] == 1:
            cy, cx = rng.integers(10, size - 10, 2); s = rng.uniform(2.5, 3.5)
            X[i] += 0.25 * np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * s ** 2))
        if rng.random() < (marker_rate if y[i] == 1 else 1 - marker_rate):
            X[i, 1:5, 1:5] += 1.0
    return np.clip(X, 0, 1), y

X_img, y_img = make_lesion_images()
cnn = small_cnn(RANDOM_STATE)
train_classifier(cnn, X_img[:2400, None].astype(float), y_img[:2400], epochs=8)
slices = {"same distribution": make_lesion_images(n=600, seed=5),
          "marker uninformative": make_lesion_images(n=600, marker_rate=0.5, seed=5),
          "marker reversed": make_lesion_images(n=600, marker_rate=0.03, seed=5)}
rows = []
for name, (Xa, ya) in slices.items():
    P = softmax_rows(cnn.forward(Xa[:, None].astype(float)))
    rows.append(dict(slice=name, accuracy=float((P.argmax(1) == ya).mean()), mean_confidence=float(P.max(1).mean())))
img_slices = pd.DataFrame(rows).set_index("slice")
print(img_slices.round(3).to_string())

In [ ]:
ax = img_slices.plot.bar(figsize=(7, 3.4), color=["#2980b9", "#c0392b"], rot=0)
ax.set_ylim(0, 1.05); ax.set_ylabel("share"); ax.set_title("accuracy and mean confidence of the network on three slices", fontsize=9)
plt.tight_layout(); plt.show()

## 4. Can the explanation be manipulated? &#91;[2](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-04/lecture.html#ref-2), [3](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-04/lecture.html#ref-3)&#93;

*Lecture: [Can the explanation be manipulated?](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-04/lecture.html#can-the-explanation-be-manipulated)*

**From the formulas to the code.** `sc.transform` standardises the measurements, so one unit is one standard deviation, the SD of the printed lines. `smlp` is a multilayer perceptron (MLP), a network of fully connected layers, and `Softplus` is the smooth version of the rectified linear unit (ReLU), $\max(0, a)$. In `explanation`, `g` is the gradient of the score $s_t$, and `x * g` is gradient times input, $\phi_j = x_j\, \partial s_t / \partial x_j$. In `attack`, `d` is the perturbation $\delta$, `eps` the budget $\varepsilon$, `lam` the constant $\lambda$ and `lr` the learning rate $\eta$. `cos` is the cosine similarity, `G` the gradient of $L$, and the last two lines of the loop are the Adam step with `np.clip`. The loop always runs its 400 steps and has no earlier stopping rule.

In [ ]:
from sklearn.preprocessing import StandardScaler

class Softplus:
    """A smooth ReLU, so that the gradient of the explanation exists everywhere."""
    def forward(self, X):
        self.X = X; return np.logaddexp(0, X)
    def backward(self, G):
        return G / (1 + np.exp(-self.X))

sc = StandardScaler().fit(Xtr); Ztr, Zte = sc.transform(Xtr), sc.transform(Xte)
rng = np.random.default_rng(RANDOM_STATE)
smlp = Net([Dense(30, 48, rng), Softplus(), Dense(48, 2, rng)])
train_classifier(smlp, Ztr, ytr.values, epochs=150, batch=64, lr=3e-3)
print(f"MLP test accuracy {float((smlp.forward(Zte).argmax(1) == yte.values).mean()):.3f}")
W1, b1, W2, b2 = smlp.layers[0].W, smlp.layers[0].b, smlp.layers[2].W, smlp.layers[2].b
sig = lambda z: 1 / (1 + np.exp(-z))

def prob(x):
    logits = np.logaddexp(0, x @ W1 + b1) @ W2 + b2
    e = np.exp(logits - logits.max()); return e / e.sum()

def explanation(x, t):
    """Gradient times input of the logit of class t, and the pieces its derivative needs."""
    s = sig(x @ W1 + b1); g = W1 @ (W2[:, t] * s)
    return x * g, g, s

def attack(x, eps=0.35, lam=60.0, steps=400, lr=0.02, seed=0):
    """Change the explanation as much as possible while keeping the prediction, within +/- eps standard deviations."""
    t = int(prob(x).argmax()); p0 = prob(x)[t]; e0 = explanation(x, t)[0]
    d = np.random.default_rng(seed).uniform(-0.01, 0.01, x.shape)   # a small random start: at d = 0 every gradient is zero
    m, v = np.zeros_like(x), np.zeros_like(x)
    for k in range(1, steps + 1):
        xp = x + d; e, g, s = explanation(xp, t)
        ne, n0 = np.linalg.norm(e), np.linalg.norm(e0); cos = e @ e0 / (ne * n0)
        dc = e0 / (ne * n0) - cos * e / ne ** 2                          # derivative of the cosine w.r.t. e
        Jg = W1 @ ((W2[:, t] * s * (1 - s))[:, None] * W1.T)             # Jacobian of the gradient: symmetric
        dcos = g * dc + Jg @ (xp * dc)
        p = prob(xp); grads = W1 @ (W2 * s[:, None])
        G = dcos + 2 * lam * (p[t] - p0) * p[t] * (grads[:, t] - grads @ p)
        m = 0.9 * m + 0.1 * G; v = 0.999 * v + 0.001 * G ** 2
        d = np.clip(d - lr * (m / (1 - 0.9 ** k)) / (np.sqrt(v / (1 - 0.999 ** k)) + 1e-8), -eps, eps)
    return d, e0, t, p0

CASE = 3; x = Zte[CASE]
delta, e0, t, p0 = attack(x)
p1 = prob(x + delta)[t]; e1 = explanation(x + delta, t)[0]
cos_after = float(e0 @ e1 / (np.linalg.norm(e0) * np.linalg.norm(e1)))
print(f"prediction {p0:.4f} -> {p1:.4f}   (change {abs(p1 - p0):.5f})")
print(f"cosine similarity of the explanations before and after: {cos_after:+.3f}")
print(f"perturbation: largest {np.abs(delta).max():.3f} SD, mean {np.abs(delta).mean():.3f} SD")

In [ ]:
b = pd.DataFrame({"before": e0, "after": e1}, index=FEATURES)
b["rank before"] = b["before"].abs().rank(ascending=False).astype(int)
b["rank after"] = b["after"].abs().rank(ascending=False).astype(int)
top = b.reindex(b["before"].abs().sort_values(ascending=False).index).head(8)
print(top.round(4).to_string())
moved = (top["rank before"] - top["rank after"]).abs()
print(f"\nmean rank movement of the original top 8 features: {moved.mean():.1f} positions")
fig, ax = plt.subplots(figsize=(7.5, 3.6))
sel = b["before"].abs().sort_values(ascending=False).index[:10][::-1]; yy = np.arange(len(sel))
ax.barh(yy - 0.19, b.loc[sel, "before"], 0.38, label="before", color="#2980b9")
ax.barh(yy + 0.19, b.loc[sel, "after"], 0.38, label="after", color="#c0392b")
ax.set_yticks(yy); ax.set_yticklabels([s[:22] for s in sel], fontsize=8); ax.axvline(0, color="grey", lw=.6)
ax.legend(fontsize=8); ax.set_title(f"Same case, prediction changed by {abs(p1 - p0):.5f}", fontsize=9)
plt.tight_layout(); plt.show()

**Exercise B.** Store the cosine similarity between the explanations before and after the attack in `cos_ab`.

In [ ]:
cos_ab = None   # your answer

check("Exercise B", cos_ab, cos_after)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
cos_ab = cos_after

check("Exercise B", cos_ab, cos_after)

## 5. Fairness and proxies &#91;[5](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-04/lecture.html#ref-5)&#93;

*Lecture: [Can the decision be trusted? Fairness and proxies](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-04/lecture.html#can-the-decision-be-trusted-fairness-and-proxies)*

**From the formulas to the code.** `pred` is the decision $\hat{y}$: 1 when the predicted probability of repaying is above 0.5. In `group_metrics`, `yhat[m].mean()` is the approval rate of a group. `TPR`, the true positive rate, is the mean of `yhat` over the applicants of the group with `y == 1`, and `FPR`, the false positive rate, is the same mean over those with `y == 0`. The two printed differences are the largest rate minus the smallest, and `np.corrcoef` returns the correlation coefficient $r$. SHAP stands for Shapley additive explanations: `shap.TreeExplainer` returns one value $\phi_j$ per applicant and feature on the scale of the log-odds, `by_group` holds the group means and `gap` the differences $\Delta_j$. The last cell trains `gb2` without `postcode_score` and computes the same measures again.

In [ ]:
p_te = models['gradient boosting'].predict_proba(Xte_c)[:, 1]
pred = (p_te > .5).astype(int)

def group_metrics(y, yhat, g):
    out = {}
    for grp in np.unique(g):
        m = (g == grp).values
        tpr = yhat[m][y[m] == 1].mean() if (y[m] == 1).sum() else np.nan
        fpr = yhat[m][y[m] == 0].mean() if (y[m] == 0).sum() else np.nan
        out[grp] = {"n": int(m.sum()), "approval rate": yhat[m].mean(),
                    "TPR": tpr, "FPR": fpr, "accuracy": (yhat[m] == y[m]).mean()}
    return pd.DataFrame(out).T

gm = group_metrics(yte_c.values, pred, gte)
print(gm.round(3).to_string())
print(f"\ndemographic parity difference : "
      f"{gm['approval rate'].max() - gm['approval rate'].min():.3f}")
print(f"equal opportunity difference  : {gm['TPR'].max() - gm['TPR'].min():.3f}")
print(f"\ncorrelation(postcode_score, group==B) : "
      f"{np.corrcoef(X_cr.postcode_score, (g_cr == 'B').astype(float))[0, 1]:.3f}")
print("the column 'group' is not an input of the model.")

In [ ]:
ax = gm[["approval rate", "TPR"]].plot.bar(figsize=(6, 3.2), rot=0, color=["#2980b9", "#e67e22"])
ax.set_title("approval rate and true positive rate by group", fontsize=9); ax.set_ylabel("rate"); plt.tight_layout(); plt.show()

In [ ]:
import shap
ex = shap.TreeExplainer(models['gradient boosting'])
SV_c = ex.shap_values(Xte_c)
SV_c = SV_c[..., 1] if SV_c.ndim == 3 else SV_c
SVdf = pd.DataFrame(SV_c, columns=X_cr.columns, index=Xte_c.index)

by_group = SVdf.groupby(gte.values).mean()
print("mean SHAP contribution by group, positive values push towards approval:\n")
print(by_group.round(4).to_string())

gap = (by_group.loc["A"] - by_group.loc["B"]).sort_values(key=abs, ascending=False)
print("\nbetween-group gap by feature:\n")
print(gap.round(4).to_string())

fig, ax = plt.subplots(figsize=(6.8, 3.2))
o = gap.iloc[::-1]
ax.barh(range(len(o)), o.values, color=["#c0392b" if abs(v) > .02 else "#95a5a6" for v in o.values])
ax.set_yticks(range(len(o))); ax.set_yticklabels(o.index, fontsize=8)
ax.axvline(0, color="grey", lw=.6)
ax.set_xlabel("mean SHAP(group A) - mean SHAP(group B)")
ax.set_title("Between-group gap by feature")
plt.tight_layout(); plt.show()

In [ ]:
# Verification: The proxy is removed and the measures are computed again. This step turns a finding into a claim.
keep = [c for c in X_cr.columns if c != "postcode_score"]
gb2 = GradientBoostingClassifier(random_state=RANDOM_STATE).fit(Xtr_c[keep], ytr_c)
pred2 = (gb2.predict_proba(Xte_c[keep])[:, 1] > .5).astype(int)
gm2 = group_metrics(yte_c.values, pred2, gte)

summary = pd.DataFrame({
    "with proxy": [roc_auc_score(yte_c, gb.predict_proba(Xte_c)[:, 1]),
                   gm["approval rate"].max() - gm["approval rate"].min(),
                   gm["TPR"].max() - gm["TPR"].min()],
    "proxy removed": [roc_auc_score(yte_c, gb2.predict_proba(Xte_c[keep])[:, 1]),
                      gm2["approval rate"].max() - gm2["approval rate"].min(),
                      gm2["TPR"].max() - gm2["TPR"].min()]},
    index=["test AUC", "demographic parity diff", "equal opportunity diff"])
print(summary.round(3).to_string())

## 6. Does the explanation beat random? &#91;[4](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-04/lecture.html#ref-4)&#93;

*Lecture: [Does the explanation beat random?](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-04/lecture.html#does-the-explanation-beat-random)*

**From the formulas to the code.** `baseline_row` is the neutral case $x'$, the medians of the training data, and `t` is the predicted class. `curve` records the probabilities $q_0, \ldots, q_K$ with $K = 15$: With `start_from_baseline=False` it replaces features of the case by medians, the deletion curve, and with `True` it fills the neutral case with real values, the insertion curve. `np.trapezoid(D, axis=1).mean() / 15` is the deletion area of the lecture, averaged over the 20 cases, and the column `faithfulness` is the insertion area minus the deletion area. `shap_attr` returns SHAP values for class 1. `lime_attr` is LIME, local interpretable model-agnostic explanations: `Zs` holds the distances $u_j$, `w` the weights $\pi_x(z)$ with the kernel width $0.75\sqrt{d}$, and `Ridge(alpha=1.0)` fits the surrogate. The line `a = a * (1 if t == 1 else -1)` turns the attributions towards the predicted class, and the impurity ranking, which has no sign, is left as it is.

In [ ]:
import time, shap
from sklearn.linear_model import Ridge
rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=1).fit(Xtr, ytr)
proba = lambda D: rf.predict_proba(pd.DataFrame(D, columns=FEATURES))
baseline_row = Xtr.median()

def curve(x, order, t, start_from_baseline):
    """Probability of the predicted class t while features are removed (deletion) or added (insertion) in order."""
    row = (baseline_row if start_from_baseline else x).copy(); src = x if start_from_baseline else baseline_row
    ps = [proba([row.values])[0, t]]
    for j in order[:15]:
        row[j] = src[j]; ps.append(proba([row.values])[0, t])
    return np.array(ps)

tex = shap.TreeExplainer(rf)
def shap_attr(x, seed=0):
    s = tex.shap_values(pd.DataFrame([x])); s = s[..., 1] if np.ndim(s) == 3 else s
    return pd.Series(np.ravel(s), index=FEATURES)

def lime_attr(x, seed=0):
    rng = np.random.default_rng(seed); sd = Xtr.std().values
    Z = rng.normal(x.values, sd, size=(600, len(x))); fz = proba(Z)[:, 1]; Zs = (Z - x.values) / sd
    w = np.exp(-(np.linalg.norm(Zs, axis=1) ** 2) / (0.75 * np.sqrt(len(x))) ** 2)
    return pd.Series(Ridge(alpha=1.0).fit(Zs, fz, sample_weight=w).coef_, index=FEATURES)

def impurity_attr(x, seed=0):
    return pd.Series(rf.feature_importances_, index=FEATURES)       # a global ranking used locally

def random_attr(x, seed=0):
    return pd.Series(np.random.default_rng(seed).normal(size=len(FEATURES)), index=FEATURES)

METHODS = {"SHAP": shap_attr, "LIME": lime_attr, "impurity (global)": impurity_attr, "random": random_attr}
CASES = list(range(0, 60, 3)); rows, mean_curves = [], {}
for name, fn in METHODS.items():
    t0 = time.time(); D, I = [], []
    for c in CASES:
        x = Xte.iloc[c]; t = int(proba([x.values])[0].argmax())
        a = fn(x, seed=c)
        if name != "impurity (global)":                              # signed methods: attribution towards the predicted class
            a = a * (1 if t == 1 else -1)
        order = a.sort_values(ascending=False).index.tolist()        # strongest support for the prediction first
        D.append(curve(x, order, t, False)); I.append(curve(x, order, t, True))
    D, I = np.array(D), np.array(I); mean_curves[name] = (D.mean(0), I.mean(0))
    rows.append(dict(method=name, deletion_AUC=float(np.trapezoid(D, axis=1).mean() / 15),
                     insertion_AUC=float(np.trapezoid(I, axis=1).mean() / 15), seconds=(time.time() - t0) / len(CASES)))
board = pd.DataFrame(rows).set_index("method")
board["faithfulness"] = board.insertion_AUC - board.deletion_AUC
print(f"{len(CASES)} test cases, 15 steps each; deletion: lower is better, insertion: higher is better\n")
print(board.round(3).to_string())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for name, (Dm, Im) in mean_curves.items():
    ax[0].plot(Dm, "o-", ms=3, label=name); ax[1].plot(Im, "o-", ms=3, label=name)
ax[0].set_title("deletion: features that support the prediction removed first", fontsize=9)
ax[1].set_title("insertion: features that support the prediction added first", fontsize=9)
for a in ax:
    a.set_xlabel("features changed"); a.set_ylabel("probability of the predicted class")
ax[1].legend(fontsize=8); plt.tight_layout(); plt.show()
print(f"margin of SHAP over random: deletion {board.loc['random', 'deletion_AUC'] - board.loc['SHAP', 'deletion_AUC']:.3f}, "
      f"insertion {board.loc['SHAP', 'insertion_AUC'] - board.loc['random', 'insertion_AUC']:.3f}")

**Exercise C.** By how much does SHAP beat the random ranking on faithfulness? Store the difference of the `faithfulness` column in `margin`.

In [ ]:
margin = None   # your answer

check("Exercise C", margin, float(board.loc["SHAP", "faithfulness"] - board.loc["random", "faithfulness"]))

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
margin = float(board.loc["SHAP", "faithfulness"] - board.loc["random", "faithfulness"])

check("Exercise C", margin, float(board.loc["SHAP", "faithfulness"] - board.loc["random", "faithfulness"]))

## 7. Your field: a reliability check

*Lecture: [Reliability in your field](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-04/lecture.html#reliability-in-your-field)*

Change `FIELD` to `chemistry`, `manufacturing`, `energy` or `civil` and run the cell again.

**From the formulas to the code.** `load_field` returns the features, the outcome and the name of the positive class of the chosen field. `ece(yb.values, pf)` is the expected calibration error of section 1 on the test part. `audit_curve` is a deletion curve over all features: `order` contains every column, so the curve has $d + 1$ points, and its area is divided by `len(dS) - 1`, which is $d$. `dS` uses the SHAP order, turned towards the predicted class, and `dR` a random order of the columns. Both are means over the first 15 test cases.

In [ ]:
FIELD = "medicine"          # choose: medicine, chemistry, manufacturing, energy or civil

def load_field(field, n=1500, seed=RANDOM_STATE):
    """A yes or no problem from one field: features, outcome and the name of the positive class."""
    rng = np.random.default_rng(seed)
    if field == "medicine":
        d = load_breast_cancer(as_frame=True); return d.data, d.target, "benign tumour"
    if field == "chemistry":
        from sklearn.datasets import load_wine
        d = load_wine(as_frame=True); return d.data, (d.target == 0).astype(int), "wine of cultivar 1"
    if field == "manufacturing":
        F = pd.DataFrame({"current_A": rng.normal(180, 20, n), "voltage_V": rng.normal(24, 2, n),
                          "speed_mm_s": rng.normal(6, 1.2, n), "gap_mm": rng.gamma(2, .4, n), "plate_temp_C": rng.normal(40, 15, n)})
        z = 0.04 * (F.current_A - 180) - 0.9 * (F.speed_mm_s - 6) - 1.8 * (F.gap_mm - .8) + 0.02 * (F.plate_temp_C - 40)
        return F, pd.Series((z + rng.normal(0, 1, n) > 0).astype(int)), "sound weld"
    if field == "energy":
        F = pd.DataFrame({g: rng.lognormal(m, .6, n) for g, m in (("H2", 3.5), ("CH4", 3.0), ("C2H2", 0.5), ("C2H4", 2.5), ("CO", 5.0))})
        z = 0.8 * np.log(F.C2H2) + 0.6 * np.log(F.H2 / 30) + 0.5 * np.log(F.C2H4 / 12) - 0.2 * np.log(F.CO / 150)
        return F, pd.Series((z + rng.normal(0, .6, n) > 0.6).astype(int)), "fault in the transformer"
    if field == "civil":
        F = pd.DataFrame({"cement": rng.uniform(150, 500, n), "water": rng.uniform(140, 230, n), "slag": rng.uniform(0, 200, n),
                          "superplasticizer": rng.uniform(0, 15, n), "age_days": rng.choice([7, 14, 28, 56, 90], n)})
        strength = (0.09 * F.cement - 0.15 * F.water + 0.04 * F.slag + 0.7 * F.superplasticizer
                    + 9 * np.log(F.age_days / 7) + rng.normal(0, 4, n))
        return F, pd.Series((strength > 30).astype(int)), "mix reaching 30 MPa"
    raise ValueError(f"unknown field: {field}")

import shap
from sklearn.datasets import load_breast_cancer
Xf, yf, positive = load_field(FIELD)
Xa, Xb, ya, yb = train_test_split(Xf, yf, test_size=0.4, stratify=yf, random_state=RANDOM_STATE)
model_f = GradientBoostingClassifier(random_state=RANDOM_STATE).fit(Xa, ya)
pf = model_f.predict_proba(Xb)[:, 1]
e_f, xs_f, ys_f = ece(yb.values, pf)
explainer_f = shap.TreeExplainer(model_f); base_f = Xa.median()
def audit_curve(order, x, t):
    row = x.copy(); ps = [model_f.predict_proba(pd.DataFrame([row]))[0, t]]
    for j in order:
        row[j] = base_f[j]; ps.append(model_f.predict_proba(pd.DataFrame([row]))[0, t])
    return np.array(ps)
dS, dR = [], []
for k in range(15):
    x = Xb.iloc[k]; t = int(model_f.predict_proba(pd.DataFrame([x]))[0].argmax())
    s = np.ravel(explainer_f.shap_values(pd.DataFrame([x]))[0]) * (1 if t == 1 else -1)
    dS.append(audit_curve(list(Xf.columns[np.argsort(-s)]), x, t))
    dR.append(audit_curve(list(np.random.default_rng(k).permutation(Xf.columns)), x, t))
dS, dR = np.mean(dS, 0), np.mean(dR, 0)
print(f"{FIELD}: test accuracy {model_f.score(Xb, yb):.3f}, expected calibration error {e_f:.3f}")
print(f"deletion area: SHAP order {np.trapezoid(dS) / (len(dS) - 1):.3f}, random order {np.trapezoid(dR) / (len(dR) - 1):.3f}")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot([0, 1], [0, 1], "k--", lw=1); ax[0].plot(xs_f, ys_f, "o-"); ax[0].set_title("reliability diagram", fontsize=9)
ax[0].set_xlabel("mean predicted probability"); ax[0].set_ylabel("observed frequency")
ax[1].plot(dS, "o-", label="SHAP order"); ax[1].plot(dR, "o-", label="random order"); ax[1].legend(fontsize=8)
ax[1].set_title("deletion: probability of the predicted class", fontsize=9); ax[1].set_xlabel("features removed")
plt.tight_layout(); plt.show()

**Exercise D.** Is SHAP more faithful than a random order for your field? Store the difference of the two deletion areas, random minus SHAP, in `gain`.

In [ ]:
gain = None   # your answer

check("Exercise D", gain, float(np.trapezoid(dR) / (len(dR) - 1) - np.trapezoid(dS) / (len(dS) - 1)))

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
gain = float(np.trapezoid(dR) / (len(dR) - 1) - np.trapezoid(dS) / (len(dS) - 1))

check("Exercise D", gain, float(np.trapezoid(dR) / (len(dR) - 1) - np.trapezoid(dS) / (len(dS) - 1)))

## 8. Going further (optional)

*Lecture: [Going further (optional)](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-04/lecture.html#going-further-optional)*

The derivative behind the attack, and the damage for five budgets.

**From the formulas to the code.** `attack_loss` is the function $L$ of the lecture: the cosine similarity plus $\lambda = 60$ times the squared change of the probability. `G` is its gradient from the formulas. In it, `g * dc` and `Jg @ (xp * dc)` are the two parts of the derivative of the explanation, `Jg` is the matrix of second derivatives, and the last term comes from the probability. `num` holds the finite differences with the step $\tau = 10^{-6}$, one for each of the 30 features. The loop then calls `attack` of section 4 for five budgets `eps` and stores the cosine similarity and the change of the probability.

In [ ]:
# The analytic derivative of the attack, checked against finite differences.
def attack_loss(xp, t, e0, p0, lam=60.0):
    e = explanation(xp, t)[0]; return e @ e0 / (np.linalg.norm(e) * np.linalg.norm(e0)) + lam * (prob(xp)[t] - p0) ** 2
x = Zte[CASE]; t = int(prob(x).argmax()); p0_ = prob(x)[t]; e0_ = explanation(x, t)[0]
xp = x + np.random.default_rng(1).normal(0, 0.2, x.shape)
e, g, s = explanation(xp, t); ne, n0 = np.linalg.norm(e), np.linalg.norm(e0_); cos = e @ e0_ / (ne * n0)
dc = e0_ / (ne * n0) - cos * e / ne ** 2; Jg = W1 @ ((W2[:, t] * s * (1 - s))[:, None] * W1.T)
p = prob(xp); grads = W1 @ (W2 * s[:, None])
G = g * dc + Jg @ (xp * dc) + 2 * 60.0 * (p[t] - p0_) * p[t] * (grads[:, t] - grads @ p)
num = np.array([(attack_loss(xp + 1e-6 * np.eye(30)[i], t, e0_, p0_) - attack_loss(xp - 1e-6 * np.eye(30)[i], t, e0_, p0_)) / 2e-6 for i in range(30)])
print(f"largest relative difference between the analytic and the numerical gradient: {np.abs(G - num).max() / np.abs(num).max():.1e}")

# How the damage grows with the perturbation budget.
rows = []
for eps in (0.05, 0.1, 0.2, 0.35, 0.5):
    d_, e0b, tb, p0b = attack(Zte[CASE], eps=eps)
    e1b = explanation(Zte[CASE] + d_, tb)[0]
    rows.append(dict(budget_sd=eps, cosine=float(e0b @ e1b / (np.linalg.norm(e0b) * np.linalg.norm(e1b))),
                     prediction_change=float(abs(prob(Zte[CASE] + d_)[tb] - p0b))))
budget = pd.DataFrame(rows); print(budget.round(4).to_string(index=False))

## Application challenges

Each challenge transfers the ideas of the day to a field of study. Choose the one closest to your programme and solve it in a copy of this notebook.

| Field | Challenge |
|---|---|
| Computer Science and Engineering | Add Integrated Gradients for the smooth network of section 5 to the benchmark of section 7 and compare it with SHAP. |
| Electronics and Communication Engineering | Build a domain classifier for a sensor dataset recorded under two conditions and report which features moved. |
| Electrical and Electronics Engineering | Recalibrate a fault classifier of your choice and report the expected calibration error with its bootstrap range before and after. |
| Mechanical Engineering | Repeat the attack of section 5 with a smaller perturbation budget and plot the cosine similarity against the budget. |
| Biomedical Engineering | Audit a clinical risk model of your choice for a disparity between two groups and look for a proxy with grouped SHAP values. |
| Civil Engineering | Design slices like those of section 4 for a crack detector whose training images were all taken in daylight. |
| Aeronautical Engineering | Run the deletion and insertion benchmark with the mean instead of the median as the replacement value and report whether the ranking of methods changes. |
| Biotechnology | Tune the kernel width of LIME in the benchmark of section 7 and find the value at which it beats the global ranking, if any. |

## After the notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-04/lab.html), tabs *Check yourself* and *Reflect and export*. The daily task and the research assignment are on the [day overview](https://github.com/utkukose/xai-veltech-NB-lecture/tree/main/days/day-04).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

</div>